In [15]:
# ============================================================
# 1. IMPORTAR LIBRERÍAS
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [16]:
# ============================================================
# 2. CARGAR DATASET DE ASOCIACIÓN
# ============================================================

df_asociacion = pd.read_csv("https://raw.githubusercontent.com/BAcost26/Datos-python/refs/heads/main/Parcial4/clave_A_asociacion.csv")

print("Primeras filas del dataset:")
display(df_asociacion.head())

print("Cantidad de filas y columnas:")
print(df_asociacion.shape)

print("Tipos de datos:")
print(df_asociacion.dtypes)

Primeras filas del dataset:


,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,A-T0001,A-C0058,2026-01-07,Limpieza,Detergente,1,App
1,A-T0001,A-C0058,2026-01-07,Panaderia,Pastelito,1,App
2,A-T0001,A-C0058,2026-01-07,Limpieza,Suavizante,1,App
3,A-T0002,A-C0044,2026-02-14,Panaderia,Pastelito,3,Web
4,A-T0003,A-C0002,2026-03-19,Lacteos,Queso,2,Web


Cantidad de filas y columnas:
(580, 7)
Tipos de datos:
transaccion_id    object
cliente_id        object
fecha             object
categoria         object
item              object
cantidad           int64
canal             object
dtype: object


In [19]:
# ============================================================
# 3. REVISIÓN DE NULOS, DUPLICADOS Y ESTRUCTURA
# ============================================================

print("Valores nulos por columna:")
print(df_asociacion.isnull().sum())

print("\nCantidad de registros duplicados:")
print(df_asociacion.duplicated().sum())

print("\nInformación general:")
df_asociacion.info()

Valores nulos por columna:
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64

Cantidad de registros duplicados:
1

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 580 entries, 0 to 579
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  580 non-null    object
 1   cliente_id      580 non-null    object
 2   fecha           580 non-null    object
 3   categoria       580 non-null    object
 4   item            580 non-null    object
 5   cantidad        580 non-null    int64 
 6   canal           579 non-null    object
dtypes: int64(1), object(6)
memory usage: 31.8+ KB


In [20]:
# ============================================================
# 4. PREPARAR DATOS PARA REGLAS DE ASOCIACIÓN
# ============================================================

# Nos interesan las transacciones y los productos comprados
df_asociacion_limpio = df_asociacion.dropna(subset=["transaccion_id", "item"])

# Crear tabla tipo canasta: filas = transacciones, columnas = productos
canasta = pd.crosstab(
    df_asociacion_limpio["transaccion_id"],
    df_asociacion_limpio["item"]
)

# Convertir cantidades a valores 1 y 0
canasta = canasta.applymap(lambda x: 1 if x > 0 else 0)

print("Formato de canasta de compras:")
display(canasta.head())

print("Tamaño de la canasta:")
print(canasta.shape)

Formato de canasta de compras:


item,Agua,Arroz,Cafe,Cloro,Crema,Detergente,Frijoles,Galletas,Jabon,Jugo,Leche,Pan,Pasta,Pastelito,Queso,Salsa,Suavizante,Te,Tortilla,Yogurt
transaccion_id,,,,,,,,,,,,,,,,,,,,
A-T0001,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0
A-T0002,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
A-T0003,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1
A-T0004,1,1,0,0,0,0,0,1,0,0,0,0,1,0,1,1,0,0,0,0
A-T0005,0,0,1,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0


Tamaño de la canasta:
(165, 20)


In [22]:
# ============================================================
# 5. APLICAR ALGORITMO APRIORI
# ============================================================

# min_support indica el porcentaje mínimo de transacciones donde aparece un producto o combinación
itemsets_frecuentes = apriori(
    canasta,
    min_support=0.03,
    use_colnames=True
)

print("Itemsets frecuentes encontrados:")
display(itemsets_frecuentes.sort_values(by="support", ascending=False).head(10))

Itemsets frecuentes encontrados:


,support,itemsets
6,0.303030,(Frijoles)
15,0.284848,(Salsa)
2,0.260606,(Cafe)
1,0.260606,(Arroz)
5,0.254545,(Detergente)
11,0.242424,(Pan)
14,0.236364,(Queso)
12,0.236364,(Pasta)
16,0.193939,(Suavizante)
87,0.163636,"(Pasta, Salsa)"


In [23]:
# ============================================================
# 6. GENERAR REGLAS DE ASOCIACIÓN
# ============================================================

reglas = association_rules(
    itemsets_frecuentes,
    metric="confidence",
    min_threshold=0.30
)

# Ordenar reglas por lift y confianza
reglas_ordenadas = reglas.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

print("10 reglas más relevantes:")
display(reglas_ordenadas[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(10))

10 reglas más relevantes:


,antecedents,consequents,support,confidence,lift
114,"(Frijoles, Cafe)","(Pan, Queso)",0.054545,0.750000,5.380435
109,"(Pan, Queso)","(Frijoles, Cafe)",0.054545,0.391304,5.380435
84,"(Pastelito, Detergente)",(Suavizante),0.036364,1.000000,5.156250
110,"(Queso, Frijoles)","(Pan, Cafe)",0.054545,0.750000,4.950000
113,"(Pan, Cafe)","(Queso, Frijoles)",0.054545,0.360000,4.950000
62,"(Frijoles, Cafe)",(Galletas),0.030303,0.416667,4.296875
89,"(Queso, Frijoles)",(Galletas),0.030303,0.416667,4.296875
64,(Galletas),"(Frijoles, Cafe)",0.030303,0.312500,4.296875
91,(Galletas),"(Queso, Frijoles)",0.030303,0.312500,4.296875
48,"(Salsa, Agua)",(Pasta),0.036364,1.000000,4.230769


In [14]:
# ============================================================
# 7. INTERPRETAR 5 REGLAS EN LENGUAJE DE NEGOCIO
# ============================================================

top_5_reglas = reglas_ordenadas.head(5)

for i, fila in top_5_reglas.iterrows():
    antecedente = list(fila["antecedents"])
    consecuente = list(fila["consequents"])

    print("Regla:")
    print(f"Si un cliente compra {antecedente}, también tiende a comprar {consecuente}.")
    print(f"Soporte: {fila['support']:.2f}")
    print(f"Confianza: {fila['confidence']:.2f}")
    print(f"Lift: {fila['lift']:.2f}")
    print("Interpretación: Esta relación puede servir para crear combos, promociones o sugerencias de compra.")
    print("-" * 80)

Regla:
Si un cliente compra ['Frijoles', 'Cafe'], también tiende a comprar ['Pan', 'Queso'].
Soporte: 0.05
Confianza: 0.75
Lift: 5.38
Interpretación: Esta relación puede servir para crear combos, promociones o sugerencias de compra.
--------------------------------------------------------------------------------
Regla:
Si un cliente compra ['Pan', 'Queso'], también tiende a comprar ['Frijoles', 'Cafe'].
Soporte: 0.05
Confianza: 0.39
Lift: 5.38
Interpretación: Esta relación puede servir para crear combos, promociones o sugerencias de compra.
--------------------------------------------------------------------------------
Regla:
Si un cliente compra ['Pastelito', 'Detergente'], también tiende a comprar ['Suavizante'].
Soporte: 0.04
Confianza: 1.00
Lift: 5.16
Interpretación: Esta relación puede servir para crear combos, promociones o sugerencias de compra.
--------------------------------------------------------------------------------
Regla:
Si un cliente compra ['Queso', 'Frijoles'], ta

In [24]:
# ============================================================
# 8. RECOMENDACIONES COMERCIALES
# ============================================================

print("RECOMENDACIONES COMERCIALES:")
print("1. Crear promociones con los productos que aparecen juntos en las reglas con mayor lift.")
print("2. Ubicar productos relacionados cerca uno del otro para aumentar la venta cruzada.")
print("3. Usar las reglas para sugerir productos complementarios en la app, web o punto de venta.")

RECOMENDACIONES COMERCIALES:
1. Crear promociones con los productos que aparecen juntos en las reglas con mayor lift.
2. Ubicar productos relacionados cerca uno del otro para aumentar la venta cruzada.
3. Usar las reglas para sugerir productos complementarios en la app, web o punto de venta.
